In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# ------------------------
# 1. Load Iris dataset
# ------------------------
iris = datasets.load_iris()
X = iris.data[:, :2]   # only take 2 features for decision boundary plot (sepal length & width)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(
   X, y, test_size=0.2, random_state=42
)
# ------------------------
# 2. Define model
# ------------------------
hidden_layer_size = 32
model = tf.keras.Sequential([
   tf.keras.layers.Dense(hidden_layer_size, activation='relu', input_shape=(X_train.shape[1],)),
   tf.keras.layers.Dense(3, activation='softmax')  # 3 classes
])
learning_rate = 0.01
epochs = 1000
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.SGD(learning_rate)
y_train_tf = tf.convert_to_tensor(y_train, dtype=tf.int32)
# ------------------------
# 3. Manual Training Loop
# ------------------------
for epoch in range(epochs):
   with tf.GradientTape() as tape:
       logits = model(X_train) 
       loss_value = loss_fn(y_train_tf, logits)
   grads = tape.gradient(loss_value, model.trainable_variables)
   optimizer.apply_gradients(zip(grads, model.trainable_variables))
   if (epoch + 1) % 100 == 0:
       preds = tf.argmax(logits, axis=1).numpy()
       acc = accuracy_score(y_train, preds)
       print(f"Epoch {epoch+1}/{epochs}, Loss: {loss_value.numpy():.4f}, Train Acc: {acc:.4f}")
# ------------------------
# 4. Evaluation
# ------------------------
y_pred = tf.argmax(model(X_test), axis=1).numpy()
print("\nTest Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
# ------------------------
# 5. Decision Boundary Plot
# ------------------------
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                    np.linspace(y_min, y_max, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = model(grid)
Z = tf.argmax(probs, axis=1).numpy().reshape(xx.shape)
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.3)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Spectral, edgecolors='k')
plt.title("Decision Boundary (Iris, first 2 features)")
plt.xlabel("Sepal length")
plt.ylabel("Sepal width")
plt.show()